In [19]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

ITERATIONS = 10
GAMES_PER_ITERATION = 32
SIMULATIONS = 50
TRAIN_STEPS = 50
BUFFER_SIZE = 5000
print("Ready. PyTorch version:", torch.__version__)


Ready. PyTorch version: 2.13.0+cpu


In [20]:
LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

def moves(s):
    return [i for i in range(9) if s[i] == 0]

def play(s, a):
    if a not in moves(s):
        raise ValueError("Choose an empty square.")
    s = s.copy()
    s[a] = 1
    return -s

def result(s):
    """Return value for the player to move, or None if ongoing."""
    for a, b, c in LINES:
        if s[a] == s[b] == s[c] == -1:
            return -1.0  # Opponent just won.
    return 0.0 if 0 not in s else None


In [21]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(9, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU()
        )
        self.pi = nn.Linear(64, 9)
        self.v = nn.Linear(64, 1)

    def forward(self, x):
        h = self.body(x)
        return self.pi(h), torch.tanh(self.v(h)).squeeze(-1)


def predict(net, s):
    legal = moves(s)
    if not legal:
        raise ValueError("No legal moves remain.")
    with torch.no_grad():
        logits, v = net(torch.tensor(s, dtype=torch.float32).unsqueeze(0))
        # Mask before softmax to keep illegal moves at zero probability.
        mask = torch.zeros(9, dtype=torch.bool)
        mask[legal] = True
        logits = logits[0].masked_fill(~mask, float("-inf"))
        p = F.softmax(logits, dim=0).numpy()
    return p, v.item()


In [22]:
class Node:
    def __init__(self, p):
        self.p = p
        self.n = 0
        self.w = 0.0
        self.kids = {}


def expand(net, node, s):
    p, v = predict(net, s)
    for a in moves(s):
        node.kids[a] = Node(p[a])
    return v


def mcts(net, s0, sims=50, c=1.5):
    if sims < 1:
        raise ValueError("sims must be at least 1.")
    if result(s0) is not None:
        raise ValueError("MCTS requires a non-terminal board.")
    root = Node(1.0)
    expand(net, root, s0)

    for _ in range(sims):
        node, s, path = root, s0, [root]
        while node.kids:
            N = math.sqrt(node.n + 1)
            a, node = max(
                node.kids.items(),
                key=lambda kv: (
                    (-kv[1].w / kv[1].n if kv[1].n else 0.0)
                    + c * kv[1].p * N / (1 + kv[1].n)
                )
            )
            s = play(s, a)
            path.append(node)

        v = result(s)
        if v is None:
            v = expand(net, node, s)
        for n in reversed(path):
            n.n += 1
            n.w += v
            v = -v

    pi = np.zeros(9, dtype=np.float64)
    for a, child in root.kids.items():
        pi[a] = child.n
    return pi / pi.sum()


In [23]:
def self_play(net, sims=50):
    s = np.zeros(9, dtype=np.float32)
    history = []
    while True:
        pi = mcts(net, s, sims=sims)
        history.append((s.copy(), pi.copy()))
        action = np.random.choice(9, p=pi)
        s = play(s, action)
        r = result(s)
        if r is not None:
            break

    examples = []
    z = -r  # Outcome for the player who made the final move.
    for state, policy in reversed(history):
        examples.append((state, policy, z))
        z = -z
    return examples


def generate_games(net, n_games, sims=50):
    net.eval()
    return [example for _ in range(n_games)
            for example in self_play(net, sims=sims)]


In [24]:
def train(net, data, steps=50):
    if not data or steps < 1:
        raise ValueError("Provide training data and at least one step.")
    net.train()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    S = torch.tensor(np.array([d[0] for d in data]), dtype=torch.float32)
    P = torch.tensor(np.array([d[1] for d in data]), dtype=torch.float32)
    Z = torch.tensor([d[2] for d in data], dtype=torch.float32)

    for _ in range(steps):
        logits, v = net(S)
        policy_loss = -(P * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
        value_loss = F.mse_loss(v, Z)
        loss = policy_loss + value_loss
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item()


In [25]:
np.random.seed(SEED)
torch.manual_seed(SEED)
net = Net()
buffer = []
loss_history = []

for it in range(ITERATIONS):
    games = generate_games(net, GAMES_PER_ITERATION, sims=SIMULATIONS)
    buffer.extend(games)
    buffer = buffer[-BUFFER_SIZE:]
    loss = train(net, buffer, steps=TRAIN_STEPS)
    loss_history.append(loss)
    print(f"iter {it + 1}: examples={len(buffer)}, loss={loss:.3f}", flush=True)


iter 1: examples=224, loss=2.523
iter 2: examples=437, loss=2.640
iter 3: examples=678, loss=2.382
iter 4: examples=910, loss=2.233
iter 5: examples=1145, loss=2.071
iter 6: examples=1364, loss=1.945
iter 7: examples=1591, loss=1.858
iter 8: examples=1819, loss=1.758
iter 9: examples=2052, loss=1.662
iter 10: examples=2315, loss=1.583
